# Day 29 — Understand the Reproduction as Research

Today is not about adding more experiments. The goal is to understand what we actually did:

**Paper → Repo → Failure → Diagnosis → Minimal Repair → Reproduction → Controlled Ablation → Evidence**

By the end, I should be able to explain what DENSE is doing, what we repaired, what the `ranking vs average` result means, and what can/cannot be claimed.


## 1. Freeze the exact experiment state

An accuracy number alone is not reproducible. A result depends on code, data, configuration, environment, randomness, and external LLM behavior:

$$
R=f(\text{code},\text{data},\text{config},\text{environment},\text{LLM},\text{randomness})
$$

So Git is not merely backup; the commit hash is part of experimental provenance.


In [ ]:
!git branch --show-current
!git rev-parse HEAD
!git status --short
!git log --oneline -6


## 2. Reconstruct DENSE end-to-end

```text
Text-Attributed Graph
        ↓
bundle_sample()
        ↓
small graph-related bundle
        ↓
batch_bundle_query()
        ↓
bundle_query()
        ↓
QueryHelper.query()
        ↓
LLM bundle label y_B
        ↓
GNN → node logits z_v
        ↓
bundle_loss()
        ↓
update GNN parameters
        ↓
bundle_resample()
        ↓
remove low-confidence node
        ↓
next training stage
        ↓
node classification
```

The LLM is **not trained**. The trainable object is the GNN:

$$
Z=f_\theta(X,A),\qquad Z\in\mathbb{R}^{N\times C}
$$

The LLM provides noisy **bundle-level supervision**.


## 3. Why bundles matter

For `neighbor` sampling, the code:

```text
random core node
      ↓
1-hop ... max_hop neighborhood
      ↓
remove core from candidate set
      ↓
randomly sample neighbors
      ↓
core + neighbors = bundle
```

Then the LLM sees the texts of the whole bundle and predicts the main category:

$$
B=\{v_1,\ldots,v_k\}\rightarrow y_B
$$

This is different from independently pseudo-labeling every node:

$$
v_i\rightarrow y_i
$$

The bundle is the bridge between graph structure and LLM semantic supervision.


## 4. `bundle_loss()` is the central bridge

With Cora, 100 bundles, bundle size 5:

```text
all node logits       [2708, 7]
bundle node IDs       [100, 5]
bundle_logits         [100, 5, 7]
                        ↑    ↑  ↑
                     bundle node class
```

### Average branch

For each bundle:

$$
\bar z_B=\frac{1}{|B|}\sum_{v\in B}z_v
$$

then:

$$
L_{CE}=CE(\bar z_B,y_B)
$$

So the code does **not** explicitly force every node to have the LLM label. It aggregates node logits first, computes a bundle-level loss, and gradients flow back into GNN parameters.


## 5. What the ranking term adds

First convert each node's logits into class confidence:

$$
p_{v,c}=\text{softmax}(z_v)_c
$$

Average within a bundle:

$$
\bar p_{B,c}=\frac{1}{|B|}\sum_{v\in B}p_{v,c}
$$

Define:

$$
p_{\text{LLM}}=\bar p_{B,y_B}
$$

and:

$$
p_{\max}=\max_c\bar p_{B,c}
$$

The implemented ranking term penalizes the case where the LLM class is not currently top-ranked:

$$
L_R=-\sum_B\min(\log p_{\text{LLM}}-\log p_{\max},0)
$$

Therefore:

```text
LLM class already top-1 → ranking penalty = 0
LLM class below top-1   → positive penalty
```

The experiment compares:

$$
L_{\text{average}}=L_{CE}
$$

against:

$$
L_{\text{ranking}}=L_{CE}+L_R
$$

That is why this is a meaningful single-variable ablation.


## 6. What refinement is doing

After a training stage, the GNN gives each node a softmax confidence.

For a bundle labeled $y_B$ by the LLM, refinement examines:

$$
p(y=y_B\mid v)
$$

and removes:

$$
v^*=\arg\min_{v\in B}p(y=y_B\mid v)
$$

Example:

```text
LLM bundle class = c

node A → 0.82
node B → 0.76
node C → 0.91
node D → 0.12  ← remove
node E → 0.69
```

With `--stages 300 100 100`:

```text
size 5
 ↓ train 300
refine
 ↓
size 4
 ↓ train 100
refine
 ↓
size 3
 ↓ train 100
evaluate
```

Hence the observed `5 → 4 → 3`.


## 7. Not every code change means “we changed DENSE”

This distinction is crucial.

```text
Compatibility fix
→ makes released code executable in the current setup.

Robustness fix
→ safely handles operational failures without changing successful behavior.

Algorithmic reconstruction
→ restores behavior required by the intended pipeline but absent/incomplete
  in the released implementation we worked from.

Experimental modification
→ intentionally changes a research variable to test a hypothesis.
```

Examples from our work:

- `None` LLM response → invalid label instead of regex crash: **robustness**.
- Retry/backoff for timeout/rate-limit failures: **robustness**.
- Repairing the released ranking-loss call so cross entropy receives its target: **reproduction repair**.
- Restoring the refinement path used by the staged pipeline: **algorithmic/reproduction reconstruction**, whose exact fidelity still needs to be stated carefully.
- `ranking` → `average`: **experimental modification**.

A research report must not mix these categories together.


## 8. Our controlled experiment

Main configuration:

```text
Dataset               Cora
Bundles               100
Bundle size           5
Sampling              neighbor
max_hop               2
LLM                    GPT-4o-2024-08-06 via API2D-compatible endpoint
GNN                    GCN
Stages                 300 / 100 / 100
lr / wd                0.001 / 0.001
Repeat                 1
Valid bundles          100%
Bundle class accuracy  76%
Refinement             5 → 4 → 3
```

Controlled variable: **loss type**

| Loss | Node accuracy |
|---|---:|
| Average | 0.6845 |
| Ranking | 0.7399 |

$$
\Delta=0.7399-0.6845=0.0554
$$

So ranking is **+5.54 percentage points in this controlled run**.


In [1]:
ranking_acc = 0.7399
average_acc = 0.6845
delta = ranking_acc - average_acc
print(f"Ranking : {ranking_acc:.4f}")
print(f"Average : {average_acc:.4f}")
print(f"Delta   : {delta:.4f} = {delta*100:.2f} percentage points")


Ranking : 0.7399
Average : 0.6845
Delta   : 0.0554 = 5.54 percentage points


## 9. What this result supports — and what it does not

Supported interpretation:

> Under the current repaired DENSE pipeline and this Cora configuration, adding the ranking objective produced higher node-classification accuracy than the average-only objective in this run.

This is consistent with the hypothesis that making the LLM-provided bundle class top-ranked provides useful supervision beyond bundle cross entropy alone.

But we should **not** say:

> “Ranking always improves DENSE by 5.54%.”

because `repeat=1`. We do not yet have run-to-run variance.

We also should not automatically call `0.7399` the official paper reproduction result. Our current execution still differs in provenance/configuration from a strict official reproduction, including the repaired code path and GCN configuration.

Think in levels:

```text
Smoke test
→ does the pipeline execute?

Controlled local experiment   ← current ranking vs average
→ can one variable be isolated?

Faithful reproduction
→ sufficiently align implementation/config/data/evaluation with paper?

Replication / extension
→ test stability or a new hypothesis across settings
```


## 10. Why 76% LLM bundle accuracy is interesting

We observed:

```text
Bundle valid rate = 100%
Bundle class acc  = 76%
```

So every LLM response was parseable, but bundle labels were not perfect.

Yet downstream node accuracy reached:

```text
ranking = 73.99%
average = 68.45%
```

This is why the pipeline should not be mentally simplified to:

```text
GPT labels nodes → GNN memorizes labels
```

A better picture is:

```text
LLM semantic supervision
        +
graph structure
        +
node representations
        +
bundle aggregation
        +
loss design
        +
refinement
        ↓
node classifier
```

The supervision is noisy and indirect.


## 11. The API failure taught a research-systems lesson

The interrupted run followed:

```text
API credit/request failure
        ↓
generate() → None
        ↓
query() assumed string
        ↓
re.findall(..., None)
        ↓
TypeError
        ↓
experiment crashes
```

After the defensive repair:

```text
invalid response
        ↓
query() → -1
        ↓
batch query can reject invalid bundle
```

This illustrates two layers of correctness:

$$
\text{Research reliability}
=
\text{algorithmic correctness}
+
\text{operational robustness}
$$

A mathematically correct method can still be hard to reproduce if API failures, malformed responses, environment differences, or cache behavior can destroy a long run.


## 12. Why raw logs are not the research record

We intentionally ignore `reproduction_logs/` in Git.

Raw logs can contain:

- huge progress bars,
- provider-specific errors,
- request IDs,
- machine paths,
- repetitive debugging output.

The useful scientific record is instead:

```text
exact commit
+ command
+ configuration
+ environment
+ result
+ known differences
+ interpretation
```

So:

$$
\text{Reproducibility package}
=
\text{Code}
+\text{Environment}
+\text{Data assumptions}
+\text{Command}
+\text{Configuration}
+\text{Result}
+\text{Provenance}
$$


## 13. What the last few days really trained

Superficially:

```text
fix bug → run → fix API → run again
```

Research-wise:

```text
Read paper
   ↓
build mental model
   ↓
follow real execution path in repo
   ↓
compare implementation with intended pipeline
   ↓
locate inconsistencies
   ↓
classify changes
   ↓
minimal repair
   ↓
smoke test
   ↓
scale experiment
   ↓
observe operational failure
   ↓
robustness repair
   ↓
controlled ablation
   ↓
interpret evidence with limitations
```

That is the real Day 29 content.


## 14. My current DENSE explanation

I should now be able to explain the method without looking at code:

1. Start from a text-attributed graph.
2. Sample small structurally related node bundles.
3. Ask an LLM for the main category of each bundle.
4. Treat that answer as noisy bundle-level supervision.
5. Train a GNN so aggregated node predictions agree with the bundle label.
6. Add ranking pressure so the LLM bundle class becomes the GNN's top-ranked bundle class.
7. After a stage, remove the node least compatible with the current bundle label.
8. Repeat training/refinement and evaluate node classification.

The core interaction is:

```text
LLM: semantic supervision from text
              ↕
Bundle: supervision bridge
              ↕
GNN: graph structure + node prediction
```


## 15. Where research questions begin

Once the code is understood, implementation choices become assumptions.

For refinement:

$$
v^*=\arg\min_{v\in B}p(y=y_B\mid v)
$$

we can now naturally ask:

```text
Why remove exactly one node?
Should refinement use uncertainty instead of raw confidence?
Should the LLM be queried again after refinement?
Should stopping be adaptive?
How sensitive is training to incorrect LLM bundle labels?
Could bundle construction itself be learned?
```

We are **not solving these today**.

The important transition is:

```text
read code
→ understand design choice
→ identify assumption
→ question assumption
→ research question
```


# Day 29 takeaway

I can now distinguish:

```text
paper method
vs released implementation
vs compatibility repair
vs robustness repair
vs experimental modification
vs evidence
vs claim
```

And trace the system end-to-end:

```text
graph/text
→ bundle sampling
→ LLM query
→ bundle supervision
→ GNN logits
→ bundle loss
→ optimization
→ refinement
→ node prediction
→ evaluation
```

Current controlled evidence:

$$
\text{Average}=0.6845,\qquad
\text{Ranking}=0.7399,\qquad
\Delta=+5.54\text{ pp}
$$

**Day 30:** turn this into a concise research progress report: what I understood, reproduced, repaired, observed, still cannot claim, and would investigate next.
